In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("../data/raw/faker_recent_matches.csv")
print(df.shape)
df.head()

In [ ]:
print(df.columns.tolist())
print()
print(df.info())
print()
print(df.isna().sum())

In [ ]:
df["win"] = df["win"].astype(int)
df["game_duration_min"] = df["game_duration"] / 60
df["game_creation"] = pd.to_datetime(df["game_creation"], unit="ms", errors="coerce")
df["kda"] = df["kda"].replace([np.inf, -np.inf], np.nan)

print(df[["game_creation", "game_duration_min", "kda", "win"]].head())

In [ ]:
summary = {
    "total_matches": len(df),
    "win_rate": df["win"].mean(),
    "avg_kills": df["kills"].mean(),
    "avg_deaths": df["deaths"].mean(),
    "avg_assists": df["assists"].mean(),
    "avg_kda": df["kda"].mean(),
    "avg_gold": df["gold_earned"].mean(),
    "avg_damage_to_champions": df["total_damage_to_champions"].mean(),
    "avg_damage_taken": df["total_damage_taken"].mean(),
    "avg_game_duration_min": df["game_duration_min"].mean()
}

summary

In [ ]:
summary_df = pd.DataFrame(summary.items(), columns=["metric", "value"])
summary_df

In [ ]:
champion_summary = (
    df.groupby("champion")
      .agg(
          matches=("match_id", "count"),
          win_rate=("win", "mean"),
          avg_kills=("kills", "mean"),
          avg_deaths=("deaths", "mean"),
          avg_assists=("assists", "mean"),
          avg_kda=("kda", "mean"),
          avg_damage=("total_damage_to_champions", "mean")
      )
      .sort_values("matches", ascending=False)
)

champion_summary.head(10)

In [ ]:
champion_summary_filtered = champion_summary[champion_summary["matches"] >= 3]
champion_summary_filtered.sort_values("win_rate", ascending=False)

In [ ]:
win_loss_summary = (
    df.groupby("win")
      .agg(
          matches=("match_id", "count"),
          avg_kills=("kills", "mean"),
          avg_deaths=("deaths", "mean"),
          avg_assists=("assists", "mean"),
          avg_kda=("kda", "mean"),
          avg_gold=("gold_earned", "mean"),
          avg_damage=("total_damage_to_champions", "mean"),
          avg_damage_taken=("total_damage_taken", "mean")
      )
)

win_loss_summary

In [ ]:
champion_counts = df["champion"].value_counts().head(10)

plt.figure(figsize=(10, 5))
champion_counts.plot(kind="bar")
plt.title("Top 10 Most Played Champions")
plt.xlabel("Champion")
plt.ylabel("Match Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
win_counts = df["win"].value_counts().sort_index()
win_counts.index = ["Loss", "Win"]

plt.figure(figsize=(6, 4))
win_counts.plot(kind="bar", color=["tomato", "seagreen"])
plt.title("Win vs Loss Count")
plt.xlabel("Result")
plt.ylabel("Match Count")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["kda"].dropna(), bins=20, edgecolor="black")
plt.title("KDA Distribution")
plt.xlabel("KDA")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
daily_matches = df.set_index("game_creation").resample("D").size()

plt.figure(figsize=(10, 5))
daily_matches.plot()
plt.title("Matches Over Time")
plt.xlabel("Date")
plt.ylabel("Number of Matches")
plt.tight_layout()
plt.show()

In [ ]:
import os
os.makedirs("../data/processed", exist_ok=True)
print("processed folder ready")

In [ ]:
champion_summary.to_csv("../data/processed/champion_summary.csv")
print("Saved to ../data/processed/champion_summary.csv")